# 데이터 로드

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager


In [ ]:
# 형태소 처리된 비공개 분석 입력 위치
file_path = os.environ['LG_VOC_DATA_DIR'] + os.sep

styler = pd.read_csv(file_path + '의류관리기.csv', encoding='utf-8-sig')
vacuum = pd.read_csv(file_path + '무선스틱청소기.csv', encoding='utf-8-sig')


## month, year 행 추가

In [ ]:
def add_year_month_columns(df):
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month

add_year_month_columns(styler)
add_year_month_columns(vacuum)


# 연관성 분석

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import font_manager
import pandas as pd
#plt.rcParams['font.family'] = font_prop.get_name()
#plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 한글 폰트 설정

font_path = os.environ.get('LG_VOC_FONT_PATH')
font_prop = font_manager.FontProperties(fname=font_path) if font_path else font_manager.FontProperties()
rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 연관 단어 함수
def coocc_word(sentences, keyword, window=1):
    coocc = dict()
    for sent in sentences:
        word_list = sent.split(' ')
        length = len(word_list)
        for i, word in enumerate(word_list):
            if word == keyword:
                for j in range(max(i - window, 0), min(i + window + 1, length)):
                    if word_list[j] in coocc:
                        coocc[word_list[j]] += 1
                    else:
                        coocc[word_list[j]] = 1
    df_occ = pd.Series(coocc, name='occ').sort_values(ascending=False)
    return df_occ

# 연관 단어 데이터프레임 변환 함수
def coocc_word_vis_nouns_with_ratio(df, keyword, num_of_result):
    sentences = df['content_nouns'].dropna().to_list()
    df_co = coocc_word(sentences, keyword).drop(keyword, errors='ignore').head(num_of_result - 1)

    total_frequency = df_co.sum() or 1

    df_result = pd.DataFrame({
        '순위': range(1, len(df_co) + 1),
        '연관단어': df_co.index,
        '빈도': df_co.values,
        '비율': (df_co.values / total_frequency) * 100
    })
    
    return df_result

# 2022년, 2024년 비중 계산 및 비중 편차 계산
def calculate_proportion_difference_fixed(df_22, df_24, df_result, keyword):
    df_22_coocc = coocc_word(df_22['content_nouns'].dropna().to_list(), keyword).drop(keyword, errors='ignore')
    total_22 = df_22_coocc.sum() or 1
    proportions_22 = df_22_coocc / total_22 * 100

    df_24_coocc = coocc_word(df_24['content_nouns'].dropna().to_list(), keyword).drop(keyword, errors='ignore')
    total_24 = df_24_coocc.sum() or 1
    proportions_24 = df_24_coocc / total_24 * 100

    df_result['비중_2022'] = df_result['연관단어'].map(proportions_22).fillna(0)
    df_result['비중_2024'] = df_result['연관단어'].map(proportions_24).fillna(0)
    df_result['언급 추이'] = df_result['비중_2024'] - df_result['비중_2022']

    return df_result


In [ ]:
def coocc_word_vis_nouns_with_ratio_and_trend(df, keyword, num_of_result, year_1, year_2,sent_path, df_name=""):
    import os

    # df_name 인자를 통해 변수 이름을 직접 전달하거나, 기본값 사용
    if not df_name:
        df_name = "df"

    sentences = df['content_nouns'].dropna().to_list()
    df_co = coocc_word(sentences, keyword).drop(keyword, errors='ignore').head(num_of_result - 1)

    if df_co.empty:
        print(f"'{keyword}'의 연관어가 없어 시각화를 건너뜁니다.")
        return pd.DataFrame(columns=['순위', '연관단어', '빈도', '비율', '언급 추이'])

    total_frequency = df_co.sum() or 1

    df_year_1 = df[pd.to_datetime(df['date']).dt.year == year_1].reset_index(drop=True)
    df_year_2 = df[pd.to_datetime(df['date']).dt.year == year_2].reset_index(drop=True)

    # 연관 단어 고정 (coocc_word_vis_nouns_with_ratio 결과 활용)
    df_year_result = coocc_word_vis_nouns_with_ratio(df, keyword=keyword, num_of_result=num_of_result)

    df_difference = calculate_proportion_difference_fixed(df_year_1, df_year_2, df_year_result, keyword)

    
    
    df_result = pd.DataFrame({
        '순위': range(1, len(df_co) + 1),
        '연관단어': df_co.index,
        '빈도': df_co.values,
        '비율': (df_co.values / total_frequency),
        '언급 추이': df_difference['언급 추이']
    })

    base_filename = f"{df_name}_{keyword}"
    plot_filename = os.path.join(sent_path, f"{base_filename}.png")
    csv_filename = os.path.join(sent_path, f"{base_filename}.csv")

    fig, ax1 = plt.subplots(figsize=(14, 7))
    bars = []
    for idx, value in enumerate(df_co.values):
        edgecolor = 'navy' if df_result['언급 추이'][idx] > 0 else 'none'
        bar = ax1.bar(df_co.index[idx], value, color='royalblue', alpha=0.7, width=0.7, edgecolor=edgecolor, linewidth=2)
        bars.append(bar)

    ax1.tick_params(axis='y', labelsize=12)
    ax1.set_title(f"'{keyword}' 연관단어 Top {num_of_result - 1}", fontsize=20, fontproperties=font_prop)
    ax2 = ax1.twinx()
    line, = ax2.plot(df_co.index, df_result['언급 추이'], color='black', marker='o', label='언급 추이', linewidth=1.5)
    ax2.tick_params(axis='y', labelcolor='black', labelsize=12)
    if len(df_result) and df_result['언급 추이'].min() != df_result['언급 추이'].max():
        y_min, y_max = df_result['언급 추이'].min(), df_result['언급 추이'].max()
        pad = (y_max - y_min) * 0.1
        ax2.set_ylim(y_min - pad, y_max + pad)

    for label in (ax1.get_xticklabels() + ax1.get_yticklabels() + ax2.get_yticklabels()):
        label.set_fontproperties(font_prop)

    legend = fig.legend([bars[0], line], ['빈도', '언급 추이(%)'], loc='upper right', fontsize=11, frameon=False, bbox_to_anchor=(0.9, 0.9))
    for text in legend.get_texts():
        text.set_fontproperties(font_prop)

    fig.tight_layout()
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    df_result.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    return df_result

## 불용어 처리

In [ ]:
def remove_stop_words(df, stop_words):
    """
    content_nouns에서 stop_words 리스트에 있는 단어를 제거하여 반환.
    
    Parameters:
    df (DataFrame): 원본 데이터프레임
    stop_words (list): 제거할 단어의 리스트
    
    Returns:
    DataFrame: 불용어가 제거된 데이터프레임
    """
    df['content_nouns'] = df['content_nouns'].fillna('').astype(str)
    df['content_nouns'] = df['content_nouns'].apply(
        lambda x: ' '.join([word for word in x.split() if word not in stop_words])
    )
    return df


In [ ]:
# content_nouns에 적용할 불용어 목록 (비워 두면 제거하지 않음)
stop_word = [

]

In [ ]:
styler = remove_stop_words(styler, stop_word)
vacuum = remove_stop_words(vacuum, stop_word)

In [ ]:
unexpected_path = os.path.join("results", "의외의키워드")
os.makedirs(unexpected_path, exist_ok=True)
# df: 의류관리기는 styler, 무선청소기는 vacuum

# df_name: 결과 파일명 앞에 붙는 이름

# keyword: 의외의 키워드 목록
for df, df_name, keywords in [(styler, "styler", ["미니", "시트"]),
                              (vacuum, "vacuum", ["핸디", "차량", "원룸"])]:
    for keyword in keywords:
        coocc_word_vis_nouns_with_ratio_and_trend(df=df, keyword=keyword, num_of_result=21, year_1=2022, year_2=2024,
                                                  sent_path=unexpected_path, df_name=df_name)
